# Calculadora de correntes — versão 3.3

Este notebook reorganiza o programa original em blocos menores, intercalando **células de código Python** com **explicações em Markdown**.

## Objetivos

Ao percorrer o notebook, será possível:

1. representar uma corrente por componentes cartesianas `u` e `v`;
2. converter `u/v` em velocidade e azimute;
3. converter velocidade e azimute em `u/v`;
4. classificar a direção em 16 setores da rosa dos ventos;
5. calcular corretamente o vetor médio de várias medições;
6. comparar a velocidade média vetorial com a média escalar das velocidades;
7. executar, ao final, o mesmo menu interativo existente no arquivo `.py`.

> **Convenção adotada:** o azimute é medido a partir do Norte, no sentido horário.

## 1. Convenções de sinal e direção

As componentes da corrente são definidas por:

| Componente | Sinal positivo | Sinal negativo |
|---|---|---|
| `u` | Leste | Oeste |
| `v` | Norte | Sul |

Para o azimute:

| Azimute | Direção |
|---:|---|
| $0^\circ$ | Norte |
| $90^\circ$ | Leste |
| $180^\circ$ | Sul |
| $270^\circ$ | Oeste |

Essa convenção difere da convenção trigonométrica mais comum, na qual o ângulo costuma ser medido a partir do eixo horizontal positivo.

## 2. Importações

In [3]:
from __future__ import annotations

from dataclasses import dataclass
import math

## 3. Estrutura de uma medição

A classe `Medicao` foi criada com `@dataclass`. Ela agrupa, em um único objeto, todas as informações relacionadas a uma medição:

- número da medição;
- componentes `u` e `v`;
- velocidade;
- azimute;
- setor da rosa dos ventos.

A anotação `@dataclass` faz o Python criar automaticamente métodos úteis, como o construtor `__init__`.

In [4]:
@dataclass
class Medicao:
    """Representa uma medição de corrente já convertida para u/v."""
    numero: int
    u: float
    v: float
    velocidade: float
    azimute: float
    setor: str

## 4. Rosa dos ventos com 16 setores

A circunferência de $360^\circ$ é dividida em 16 setores. Cada setor possui largura angular de:

$$
\frac{360^\circ}{16}=22{,}5^\circ
$$

A constante `EPS` representa uma tolerância numérica muito pequena. Ela é usada para tratar valores que, devido ao arredondamento de ponto flutuante, aparecem como algo próximo de zero.

In [5]:
SETORES_16 = [
    ("N", "Norte", "↑"),
    ("NNE", "Norte-Nordeste", "↑↗"),
    ("NE", "Nordeste", "↗"),
    ("ENE", "Leste-Nordeste", "↗→"),
    ("E", "Leste", "→"),
    ("ESE", "Leste-Sudeste", "→↘"),
    ("SE", "Sudeste", "↘"),
    ("SSE", "Sul-Sudeste", "↘↓"),
    ("S", "Sul", "↓"),
    ("SSW", "Sul-Sudoeste", "↓↙"),
    ("SW", "Sudoeste", "↙"),
    ("WSW", "Oeste-Sudoeste", "↙←"),
    ("W", "Oeste", "←"),
    ("WNW", "Oeste-Noroeste", "←↖"),
    ("NW", "Noroeste", "↖"),
    ("NNW", "Norte-Noroeste", "↖↑"),
]


EPS = 1e-12

## 5. Funções auxiliares para ângulos e zeros numéricos

### Normalização do azimute

A operação módulo mantém o azimute no intervalo:

$$
0^\circ \leq \theta < 360^\circ
$$

Assim, por exemplo:

- $370^\circ \rightarrow 10^\circ$;
- $-10^\circ \rightarrow 350^\circ$.

### Limpeza de valores próximos de zero

Funções trigonométricas podem produzir valores como `-6.12e-17` quando o resultado matemático esperado é zero. A função `limpar_zero` evita que esses resíduos apareçam na saída.

In [6]:
def normalizar_azimute(azimute_graus: float) -> float:
    """Normaliza o azimute para o intervalo [0, 360)."""
    return azimute_graus % 360.0

def limpar_zero(valor: float) -> float:
    """Evita saída visual do tipo -0.0000."""
    if abs(valor) < EPS:
        return 0.0
    return valor

In [7]:
# Exemplos de normalização
for angulo in [370, -10, 720, 90]:
    print(f"{angulo:>4}° -> {normalizar_azimute(angulo):.1f}°")

 370° -> 10.0°
 -10° -> 350.0°
 720° -> 0.0°
  90° -> 90.0°


## 6. Conversão de velocidade e azimute para `u/v`

Se $V$ é a velocidade e $\theta$ é o azimute, então:

$$
u = V\,\sin(\theta)
$$

$$
v = V\,\cos(\theta)
$$

O seno aparece em `u` e o cosseno em `v` porque o ângulo é medido a partir do Norte, isto é, a partir do eixo associado à componente `v`.

In [8]:
def vel_az_para_uv(velocidade: float, azimute_graus: float) -> tuple[float, float]:
    """
    Converte velocidade + azimute para componentes u/v.

    Como o azimute é medido a partir do Norte:
    - u = velocidade * sen(azimute)
    - v = velocidade * cos(azimute)
    """
    az_rad = math.radians(normalizar_azimute(azimute_graus))

    u = velocidade * math.sin(az_rad)
    v = velocidade * math.cos(az_rad)

    return limpar_zero(u), limpar_zero(v)

## 7. Conversão de `u/v` para velocidade e azimute

O módulo do vetor é calculado pelo teorema de Pitágoras:

$$
V=\sqrt{u^2+v^2}
$$

No Python, `math.hypot(u, v)` calcula esse valor.

O azimute é obtido por:

$$
\theta=\operatorname{atan2}(u,v)
$$

A ordem `atan2(u, v)` é intencional. A função precisa considerar que o ângulo geográfico é contado a partir do Norte, e não a partir do eixo horizontal positivo.

In [9]:
def uv_para_vel_az(u: float, v: float) -> tuple[float, float]:
    """
    Converte componentes u/v para velocidade + azimute.

    Usa atan2(u, v), e não atan2(v, u), porque o azimute é contado
    a partir do Norte, no sentido horário.
    """
    velocidade = math.hypot(u, v)

    if velocidade < EPS:
        return 0.0, 0.0

    azimute = math.degrees(math.atan2(u, v))
    azimute = normalizar_azimute(azimute)

    return limpar_zero(velocidade), limpar_zero(azimute)

### Conferência com as quatro direções cardeais

In [10]:
testes_cardinais = [
    (1.0, 0.0, "Leste"),
    (0.0, 1.0, "Norte"),
    (-1.0, 0.0, "Oeste"),
    (0.0, -1.0, "Sul"),
]

for u, v, direcao in testes_cardinais:
    velocidade, azimute = uv_para_vel_az(u, v)
    print(
        f"{direcao:>6}: u={u:>4.1f}, v={v:>4.1f}, "
        f"velocidade={velocidade:.1f}, azimute={azimute:.1f}°"
    )

 Leste: u= 1.0, v= 0.0, velocidade=1.0, azimute=90.0°
 Norte: u= 0.0, v= 1.0, velocidade=1.0, azimute=0.0°
 Oeste: u=-1.0, v= 0.0, velocidade=1.0, azimute=270.0°
   Sul: u= 0.0, v=-1.0, velocidade=1.0, azimute=180.0°


## 8. Classificação em 16 setores

Para localizar o setor, o programa soma metade da largura de um setor, $11{,}25^\circ$, e divide o resultado por $22{,}5^\circ$.

Isso faz com que cada azimute seja associado ao setor cujo centro está mais próximo dele.

In [11]:
def setor_16(azimute_graus: float) -> tuple[str, str, str]:
    """Retorna abreviação, nome e seta textual para o azimute em 16 setores."""
    az = normalizar_azimute(azimute_graus)
    indice = int((az + 11.25) // 22.5) % 16
    return SETORES_16[indice]

def texto_setor_16(azimute_graus: float) -> str:
    """Retorna texto completo do setor de direção."""
    sigla, nome, seta = setor_16(azimute_graus)
    return f"{sigla} - {nome} {seta}"

In [12]:
for azimute in [0, 22.5, 45, 90, 180, 270, 350]:
    print(f"{azimute:>6.1f}° -> {texto_setor_16(azimute)}")

   0.0° -> N - Norte ↑
  22.5° -> NNE - Norte-Nordeste ↑↗
  45.0° -> NE - Nordeste ↗
  90.0° -> E - Leste →
 180.0° -> S - Sul ↓
 270.0° -> W - Oeste ←
 350.0° -> N - Norte ↑


## 9. Formatação e leitura de dados

As funções desta seção tratam a interação com o usuário:

- `fmt`: apresenta números com vírgula decimal;
- `ler_float`: aceita ponto ou vírgula na entrada;
- `ler_float_ou_comando`: aceita números e comandos como `fim`;
- `ler_opcao`: valida uma escolha de menu.

Em um notebook, funções baseadas em `input()` funcionam, mas deixam a célula aguardando a digitação do usuário. Por isso, os exemplos didáticos anteriores usam valores definidos diretamente no código.

In [13]:
def fmt(valor: float, casas: int = 4) -> str:
    """Formata número com vírgula decimal para leitura no terminal."""
    valor = limpar_zero(valor)
    return f"{valor:.{casas}f}".replace(".", ",")

def ler_float(mensagem: str, minimo: float | None = None) -> float:
    """
    Lê número aceitando vírgula ou ponto decimal.

    Exemplo:
    - 0,165
    - 0.165
    """
    while True:
        texto = input(mensagem).strip().replace(",", ".")

        try:
            valor = float(texto)
        except ValueError:
            print("Entrada inválida. Digite um número.")
            continue

        if minimo is not None and valor < minimo:
            print(f"Valor inválido. Digite um número maior ou igual a {fmt(minimo)}.")
            continue

        return valor

def ler_float_ou_comando(
    mensagem: str,
    minimo: float | None = None,
    comandos_validos: set[str] | None = None,
) -> float | str:
    """
    Lê número aceitando vírgula ou ponto decimal.

    Também aceita comandos textuais definidos em comandos_validos.
    Retorna float quando o usuário digita número.
    Retorna str quando o usuário digita comando.
    """
    if comandos_validos is None:
        comandos_validos = {"fim"}

    comandos_texto = ", ".join(f"'{cmd}'" for cmd in sorted(comandos_validos))

    while True:
        texto_original = input(mensagem).strip()
        texto_minusculo = texto_original.lower()

        if texto_minusculo in comandos_validos:
            return texto_minusculo

        texto = texto_original.replace(",", ".")

        try:
            valor = float(texto)
        except ValueError:
            print(f"Entrada inválida. Digite um número ou {comandos_texto}.")
            continue

        if minimo is not None and valor < minimo:
            print(f"Valor inválido. Digite um número maior ou igual a {fmt(minimo)}.")
            continue

        return valor

def ler_opcao(mensagem: str, opcoes_validas: set[str]) -> str:
    """Lê uma opção de menu."""
    while True:
        opcao = input(mensagem).strip()

        if opcao in opcoes_validas:
            return opcao

        print(f"Opção inválida. Escolha uma destas opções: {', '.join(sorted(opcoes_validas))}.")

## 10. Interpretação física dos sinais e apresentação dos resultados

In [14]:
def texto_sentido_u(u: float) -> str:
    """Indica o sentido físico da componente u."""
    u = limpar_zero(u)

    if u > 0:
        return "Leste"

    if u < 0:
        return "Oeste"

    return "sem componente Leste/Oeste"

def texto_sentido_v(v: float) -> str:
    """Indica o sentido físico da componente v."""
    v = limpar_zero(v)

    if v > 0:
        return "Norte"

    if v < 0:
        return "Sul"

    return "sem componente Norte/Sul"

def imprimir_resultado_uv_para_vel_az(
    u: float,
    v: float,
    titulo: str = "Resultado",
) -> None:
    """Imprime resultado completo a partir das componentes u/v."""
    velocidade, azimute = uv_para_vel_az(u, v)
    sigla, nome, seta = setor_16(azimute)

    print("\n" + "=" * 60)
    print(titulo)
    print("=" * 60)
    print(f"u: {fmt(u)}  ({texto_sentido_u(u)})")
    print(f"v: {fmt(v)}  ({texto_sentido_v(v)})")
    print(f"Velocidade: {fmt(velocidade)}")
    print(f"Azimute: {fmt(azimute, 2)}°")
    print(f"Direção aproximada: {sigla} - {nome} {seta}")
    print("=" * 60)

def imprimir_resultado_vel_az_para_uv(
    velocidade: float,
    azimute: float,
    titulo: str = "Resultado",
) -> None:
    """Imprime resultado completo a partir de velocidade + azimute."""
    azimute = normalizar_azimute(azimute)
    u, v = vel_az_para_uv(velocidade, azimute)
    sigla, nome, seta = setor_16(azimute)

    print("\n" + "=" * 60)
    print(titulo)
    print("=" * 60)
    print(f"Velocidade: {fmt(velocidade)}")
    print(f"Azimute: {fmt(azimute, 2)}°")
    print(f"Direção aproximada: {sigla} - {nome} {seta}")
    print(f"u: {fmt(u)}  ({texto_sentido_u(u)})")
    print(f"v: {fmt(v)}  ({texto_sentido_v(v)})")
    print("=" * 60)

### Exemplo direto: velocidade e azimute para `u/v`

In [15]:
imprimir_resultado_vel_az_para_uv(
    velocidade=1.0,
    azimute=45.0,
    titulo="Exemplo: corrente para Nordeste",
)


Exemplo: corrente para Nordeste
Velocidade: 1,0000
Azimute: 45,00°
Direção aproximada: NE - Nordeste ↗
u: 0,7071  (Leste)
v: 0,7071  (Norte)


## 11. Opções 1 e 2 do menu

Estas funções reúnem as etapas de leitura, cálculo e impressão para as duas conversões básicas.

Elas não fazem cálculos novos: apenas coordenam funções já definidas.

In [16]:
def opcao_1_uv_para_vel_az() -> None:
    """Opção 1: converte u/v em velocidade + azimute."""
    print("\nOpção 1 - Converter u/v em velocidade + azimute")
    print("Convenção: u > 0 = Leste; v > 0 = Norte.")

    u = ler_float("Digite u: ")
    v = ler_float("Digite v: ")

    imprimir_resultado_uv_para_vel_az(
        u,
        v,
        "Resultado - u/v para velocidade + azimute",
    )

def opcao_2_vel_az_para_uv() -> None:
    """Opção 2: converte velocidade + azimute em u/v."""
    print("\nOpção 2 - Converter velocidade + azimute em u/v")
    print("Convenção: 0° = Norte; 90° = Leste; 180° = Sul; 270° = Oeste.")

    velocidade = ler_float("Digite a velocidade: ", minimo=0.0)
    azimute = ler_float("Digite o azimute em graus: ")

    imprimir_resultado_vel_az_para_uv(
        velocidade,
        azimute,
        "Resultado - velocidade + azimute para u/v",
    )

## 12. Organização da lista de medições

As funções seguintes permitem:

- imprimir as medições em forma de tabela;
- remover a última medição;
- adicionar uma nova medição à lista.

A lista contém objetos da classe `Medicao`.

In [17]:
def imprimir_tabela_medicoes(medicoes: list[Medicao], formato: str) -> None:
    """Imprime tabela simples com as medições digitadas."""
    if not medicoes:
        print("\nAinda não há medições registradas.")
        return

    print("\nMedições usadas no cálculo:")

    if formato == "1":
        print("-" * 76)
        print(f"{'N':>3} | {'u':>12} | {'v':>12} | {'velocidade':>12} | {'azimute':>10} | direção")
        print("-" * 76)

        for medicao in medicoes:
            print(
                f"{medicao.numero:>3} | "
                f"{fmt(medicao.u):>12} | "
                f"{fmt(medicao.v):>12} | "
                f"{fmt(medicao.velocidade):>12} | "
                f"{fmt(medicao.azimute, 2):>10}° | "
                f"{medicao.setor}"
            )

        print("-" * 76)

    else:
        print("-" * 76)
        print(f"{'N':>3} | {'velocidade':>12} | {'azimute':>10} | {'u':>12} | {'v':>12} | direção")
        print("-" * 76)

        for medicao in medicoes:
            print(
                f"{medicao.numero:>3} | "
                f"{fmt(medicao.velocidade):>12} | "
                f"{fmt(medicao.azimute, 2):>10}° | "
                f"{fmt(medicao.u):>12} | "
                f"{fmt(medicao.v):>12} | "
                f"{medicao.setor}"
            )

        print("-" * 76)

def desfazer_ultima_medicao(medicoes: list[Medicao]) -> None:
    """Remove a última medição digitada."""
    if not medicoes:
        print("\nNão há medição para desfazer.")
        return

    removida = medicoes.pop()

    print(
        "\nÚltima medição removida: "
        f"N={removida.numero}, "
        f"u={fmt(removida.u)}, "
        f"v={fmt(removida.v)}, "
        f"vel={fmt(removida.velocidade)}, "
        f"az={fmt(removida.azimute, 2)}°"
    )

    for indice, medicao in enumerate(medicoes, start=1):
        medicao.numero = indice

def adicionar_medicao(
    medicoes: list[Medicao],
    u: float,
    v: float,
    velocidade: float,
    azimute: float,
) -> None:
    """Adiciona uma medição à lista e imprime conferência."""
    azimute = normalizar_azimute(azimute)
    setor_texto = texto_setor_16(azimute)

    medicao = Medicao(
        numero=len(medicoes) + 1,
        u=u,
        v=v,
        velocidade=velocidade,
        azimute=azimute,
        setor=setor_texto,
    )

    medicoes.append(medicao)

    print(
        f"  Conferência: u={fmt(u)}, v={fmt(v)}, "
        f"vel={fmt(velocidade)}, az={fmt(azimute, 2)}° "
        f"({setor_texto})"
    )

## 13. Média vetorial de correntes

Para $n$ medições, calcula-se primeiro a média de cada componente:

$$
\overline{u}=\frac{1}{n}\sum_{i=1}^{n}u_i
$$

$$
\overline{v}=\frac{1}{n}\sum_{i=1}^{n}v_i
$$

Depois, o vetor médio é obtido por:

$$
\left|\overline{\vec{V}}\right|
=\sqrt{\overline{u}^{\,2}+\overline{v}^{\,2}}
$$

O azimute médio é calculado a partir de $\overline{u}$ e $\overline{v}$.

### Por que não calcular a média aritmética dos azimutes?

Os azimutes $350^\circ$ e $10^\circ$ apontam quase para o Norte. A média aritmética seria:

$$
\frac{350^\circ+10^\circ}{2}=180^\circ
$$

Esse resultado apontaria para o Sul e seria fisicamente incorreto. A decomposição em componentes resolve o problema circular dos ângulos.

In [18]:
def calcular_e_imprimir_vetor_medio(
    medicoes: list[Medicao],
    formato: str,
) -> None:
    """Calcula e imprime o vetor médio a partir das medições."""
    quantidade = len(medicoes)

    if quantidade == 0:
        print("\nNenhuma medição foi informada. Nada foi calculado.")
        return

    soma_u = sum(medicao.u for medicao in medicoes)
    soma_v = sum(medicao.v for medicao in medicoes)

    media_u = soma_u / quantidade
    media_v = soma_v / quantidade

    velocidade_media_vetorial, azimute_medio = uv_para_vel_az(media_u, media_v)
    velocidade_media_escalar = sum(medicao.velocidade for medicao in medicoes) / quantidade

    imprimir_tabela_medicoes(medicoes, formato)

    imprimir_resultado_uv_para_vel_az(
        media_u,
        media_v,
        "Resultado - vetor médio das medições",
    )

    print("\nResumo estatístico:")
    print(f"Número de medições: {quantidade}")
    print(f"Soma de u: {fmt(soma_u)}")
    print(f"Soma de v: {fmt(soma_v)}")
    print(f"Média de u: {fmt(media_u)}")
    print(f"Média de v: {fmt(media_v)}")
    print(f"Velocidade média vetorial: {fmt(velocidade_media_vetorial)}")
    print(f"Azimute do vetor médio: {fmt(azimute_medio, 2)}°")
    print(f"Média escalar das velocidades: {fmt(velocidade_media_escalar)}")

    if velocidade_media_escalar > EPS:
        razao_vetorial_escalar = velocidade_media_vetorial / velocidade_media_escalar

        print(f"Razão vetorial/escalar: {fmt(razao_vetorial_escalar)}")

        print("\nInterpretação da razão vetorial/escalar:")
        print(
            "A razão vetorial/escalar indica o quanto as direções das medições "
            "estão coerentes entre si."
        )
        print(
            "Ela é calculada como: velocidade média vetorial dividida pela "
            "média escalar das velocidades."
        )
        print(
            "Valor próximo de 1: as correntes apontam quase todas para a mesma direção."
        )
        print(
            "Valor próximo de 0: as correntes se cancelam, indicando direções opostas "
            "ou muito dispersas."
        )
        print(
            "Em correntes influenciadas por maré, uma razão baixa pode indicar "
            "alternância entre sentidos de enchente e vazante."
        )

### Exemplo: medições de $350^\circ$ e $10^\circ$

As duas velocidades serão consideradas iguais a `1.0`. O vetor médio deve permanecer próximo do Norte.

In [19]:
medicoes_exemplo = []

for azimute in [350.0, 10.0]:
    velocidade = 1.0
    u, v = vel_az_para_uv(velocidade, azimute)
    adicionar_medicao(medicoes_exemplo, u, v, velocidade, azimute)

calcular_e_imprimir_vetor_medio(medicoes_exemplo, formato="2")

  Conferência: u=-0,1736, v=0,9848, vel=1,0000, az=350,00° (N - Norte ↑)
  Conferência: u=0,1736, v=0,9848, vel=1,0000, az=10,00° (N - Norte ↑)

Medições usadas no cálculo:
----------------------------------------------------------------------------
  N |   velocidade |    azimute |            u |            v | direção
----------------------------------------------------------------------------
  1 |       1,0000 |     350,00° |      -0,1736 |       0,9848 | N - Norte ↑
  2 |       1,0000 |      10,00° |       0,1736 |       0,9848 | N - Norte ↑
----------------------------------------------------------------------------

Resultado - vetor médio das medições
u: 0,0000  (sem componente Leste/Oeste)
v: 0,9848  (Norte)
Velocidade: 0,9848
Azimute: 360,00°
Direção aproximada: N - Norte ↑

Resumo estatístico:
Número de medições: 2
Soma de u: 0,0000
Soma de v: 1,9696
Média de u: 0,0000
Média de v: 0,9848
Velocidade média vetorial: 0,9848
Azimute do vetor médio: 360,00°
Média escalar das velo

## 14. Razão vetorial/escalar

A média escalar das velocidades é:

$$
\overline{V}_{\mathrm{escalar}}
=\frac{1}{n}\sum_{i=1}^{n}V_i
$$

A razão usada pelo programa é:

$$
R=\frac{\left|\overline{\vec{V}}\right|}
{\overline{V}_{\mathrm{escalar}}}
$$

Interpretação geral:

- $R$ próximo de `1`: direções persistentes e semelhantes;
- $R$ intermediário: alguma dispersão direcional;
- $R$ próximo de `0`: forte cancelamento entre direções opostas ou muito dispersas.

Em séries influenciadas pela maré, um valor baixo pode refletir alternância entre enchente e vazante.

## 15. Entrada manual de várias medições

A opção 3 mantém o fluxo do programa de terminal:

- o usuário escolhe o formato de entrada;
- cada medição é adicionada individualmente;
- `listar` mostra as medições registradas;
- `desfazer` remove a última medição;
- `fim` encerra a entrada e calcula o vetor médio.

A célula abaixo apenas define a função. Ela não inicia automaticamente a entrada interativa.

In [20]:
def opcao_3_vetor_medio_manual() -> None:
    """
    Opção 3: calcula vetor médio a partir de várias medições.

    Na v3.2:
    - O usuário não informa previamente a quantidade.
    - Digita as medições uma a uma.
    - Encerra digitando 'fim'.
    - Pode apagar a última medição digitando 'desfazer'.
    - Pode listar as medições digitando 'listar'.

    Importante:
    - O azimute médio não é calculado pela média aritmética dos azimutes.
    - Primeiro cada medição é convertida para u/v.
    - Depois calcula-se média_u e média_v.
    - Por fim, média_u/média_v são convertidos para velocidade + azimute.
    """
    print("\nOpção 3 - Calcular vetor médio de várias medições")
    print("Sem CSV: as medições serão digitadas manualmente.")
    print("Comandos disponíveis no primeiro campo:")
    print("- fim: calcula com as medições já digitadas")
    print("- desfazer: apaga a última medição digitada")
    print("- listar: mostra as medições já digitadas")

    print("\nEscolha o formato das medições:")
    print("1 - Cada medição como u/v")
    print("2 - Cada medição como velocidade + azimute")

    formato = ler_opcao("Formato escolhido: ", {"1", "2"})

    medicoes: list[Medicao] = []

    while True:
        numero_medicao = len(medicoes) + 1

        print(f"\nMedição {numero_medicao}")
        print("Digite 'fim', 'desfazer' ou 'listar' no primeiro campo.")

        if formato == "1":
            u_lido = ler_float_ou_comando(
                "  u: ",
                comandos_validos={"fim", "desfazer", "listar"},
            )

            if isinstance(u_lido, str):
                if u_lido == "fim":
                    break
                if u_lido == "desfazer":
                    desfazer_ultima_medicao(medicoes)
                    continue
                if u_lido == "listar":
                    imprimir_tabela_medicoes(medicoes, formato)
                    continue

            v_lido = ler_float_ou_comando(
                "  v: ",
                comandos_validos={"fim"},
            )

            if isinstance(v_lido, str):
                print("Entrada encerrada. A medição incompleta não será usada.")
                break

            u = u_lido
            v = v_lido
            velocidade, azimute = uv_para_vel_az(u, v)

            adicionar_medicao(medicoes, u, v, velocidade, azimute)

        else:
            velocidade_lida = ler_float_ou_comando(
                "  velocidade: ",
                minimo=0.0,
                comandos_validos={"fim", "desfazer", "listar"},
            )

            if isinstance(velocidade_lida, str):
                if velocidade_lida == "fim":
                    break
                if velocidade_lida == "desfazer":
                    desfazer_ultima_medicao(medicoes)
                    continue
                if velocidade_lida == "listar":
                    imprimir_tabela_medicoes(medicoes, formato)
                    continue

            azimute_lido = ler_float_ou_comando(
                "  azimute em graus: ",
                comandos_validos={"fim"},
            )

            if isinstance(azimute_lido, str):
                print("Entrada encerrada. A medição incompleta não será usada.")
                break

            velocidade = velocidade_lida
            azimute = normalizar_azimute(azimute_lido)
            u, v = vel_az_para_uv(velocidade, azimute)

            adicionar_medicao(medicoes, u, v, velocidade, azimute)

    calcular_e_imprimir_vetor_medio(medicoes, formato)

## 16. Ajuda conceitual do programa

In [21]:
def opcao_4_explicar_conceitos() -> None:
    """Explica os conceitos usados na calculadora de correntes."""
    print("\n" + "=" * 72)
    print("Conceitos usados na calculadora de correntes")
    print("=" * 72)

    print("\n1) Componentes u e v")
    print(
        "A corrente pode ser representada por duas componentes ortogonais: "
        "u e v."
    )
    print("- u representa a componente Leste/Oeste.")
    print("- v representa a componente Norte/Sul.")
    print("Nesta calculadora:")
    print("- u > 0 indica movimento para Leste.")
    print("- u < 0 indica movimento para Oeste.")
    print("- v > 0 indica movimento para Norte.")
    print("- v < 0 indica movimento para Sul.")

    print("\n2) Velocidade e azimute")
    print(
        "A velocidade é o módulo do vetor de corrente. O azimute indica "
        "a direção do vetor em graus."
    )
    print("A convenção adotada é:")
    print("- 0° = Norte")
    print("- 90° = Leste")
    print("- 180° = Sul")
    print("- 270° = Oeste")
    print(
        "Essa é uma convenção usual para representar direções geográficas, "
        "medidas a partir do Norte no sentido horário."
    )

    print("\n3) Conversão de velocidade + azimute para u/v")
    print("Como o azimute é medido a partir do Norte:")
    print("- u = velocidade × sen(azimute)")
    print("- v = velocidade × cos(azimute)")
    print(
        "Por isso, uma corrente com azimute 90° tem componente u positiva "
        "e componente v aproximadamente zero."
    )

    print("\n4) Conversão de u/v para velocidade + azimute")
    print("A velocidade é calculada por:")
    print("- velocidade = raiz(u² + v²)")
    print(
        "O azimute é calculado com atan2(u, v), e não atan2(v, u), "
        "porque o ângulo é contado a partir do Norte, não a partir do eixo x."
    )

    print("\n5) Média vetorial")
    print(
        "Para calcular uma corrente média, o programa calcula a média das "
        "componentes u e v."
    )
    print("O procedimento é:")
    print("- converter cada medição para u/v;")
    print("- calcular u médio e v médio;")
    print("- converter u médio e v médio para velocidade + azimute.")
    print(
        "Esse é o método adequado porque preserva o caráter vetorial da "
        "corrente."
    )

    print("\n6) Por que não usar média simples dos azimutes?")
    print(
        "Azimutes são ângulos circulares. Por isso, a média aritmética simples "
        "pode produzir resultados errados."
    )
    print("Exemplo:")
    print("- Uma medição com azimute 350° aponta quase para Norte.")
    print("- Outra medição com azimute 10° também aponta quase para Norte.")
    print(
        "A média aritmética simples seria 180°, que aponta para Sul. "
        "Fisicamente isso está errado."
    )
    print(
        "Ao usar u/v, o programa evita esse problema e calcula a direção média "
        "corretamente."
    )

    print("\n7) Média escalar das velocidades")
    print(
        "A média escalar das velocidades é a média simples dos módulos das "
        "velocidades medidas."
    )
    print(
        "Ela informa a intensidade média das correntes, sem considerar o "
        "cancelamento entre direções opostas."
    )

    print("\n8) Velocidade média vetorial")
    print(
        "A velocidade média vetorial é o módulo do vetor médio calculado a "
        "partir de u médio e v médio."
    )
    print(
        "Ela considera tanto a intensidade quanto a direção das medições. "
        "Por isso, correntes em sentidos opostos podem se cancelar."
    )

    print("\n9) Razão vetorial/escalar")
    print(
        "A razão vetorial/escalar é calculada como velocidade média vetorial "
        "dividida pela média escalar das velocidades."
    )
    print("Interpretação:")
    print("- valor próximo de 1: corrente com direção persistente e bem definida;")
    print("- valor intermediário: corrente com alguma variação direcional;")
    print(
        "- valor próximo de 0: forte cancelamento entre direções opostas "
        "ou muito dispersas."
    )
    print(
        "Em correntes influenciadas por maré, uma razão baixa pode indicar "
        "alternância entre sentidos de enchente e vazante."
    )

    print("\n10) Leitura oceanográfica do resultado")
    print(
        "O vetor médio representa a tendência resultante no período analisado. "
        "A média escalar representa a intensidade média do movimento, mesmo "
        "quando há reversões de direção."
    )
    print(
        "Assim, duas situações podem ter a mesma média escalar, mas vetores "
        "médios muito diferentes se as direções forem mais ou menos persistentes."
    )

    print("\n" + "=" * 72)

## 17. Menu principal

As funções abaixo preservam o menu original do arquivo `.py`.

Para iniciar o programa dentro do notebook, execute a última célula depois de retirar o comentário de `main()`.

> Durante a execução, o kernel ficará aguardando as entradas solicitadas por `input()`. Para interromper uma execução travada, use o botão **Interromper** do Jupyter ou do VS Code.

In [22]:
def mostrar_menu() -> None:
    """Mostra o menu principal."""
    print("\n" + "=" * 60)
    print("Calculadora de correntes - v3.3")
    print("=" * 60)
    print("1 - Converter u/v em velocidade + azimute")
    print("2 - Converter velocidade + azimute em u/v")
    print("3 - Calcular vetor médio de várias medições manuais")
    print("4 - Explicar conceitos usados no programa")
    print("0 - Sair")

def main() -> None:
    """Loop principal do programa."""
    while True:
        mostrar_menu()

        opcao = ler_opcao("Escolha uma opção: ", {"0", "1", "2", "3", "4"})

        if opcao == "0":
            print("\nEncerrado.")
            break

        if opcao == "1":
            opcao_1_uv_para_vel_az()

        elif opcao == "2":
            opcao_2_vel_az_para_uv()

        elif opcao == "3":
            opcao_3_vetor_medio_manual()
        elif opcao == "4":
            opcao_4_explicar_conceitos()

In [24]:
# Retire o símbolo # da linha seguinte para iniciar o menu interativo.
main()


Calculadora de correntes - v3.3
1 - Converter u/v em velocidade + azimute
2 - Converter velocidade + azimute em u/v
3 - Calcular vetor médio de várias medições manuais
4 - Explicar conceitos usados no programa
0 - Sair

Opção 1 - Converter u/v em velocidade + azimute
Convenção: u > 0 = Leste; v > 0 = Norte.

Resultado - u/v para velocidade + azimute
u: 1,2000  (Leste)
v: 0,5700  (Norte)
Velocidade: 1,3285
Azimute: 64,59°
Direção aproximada: ENE - Leste-Nordeste ↗→

Calculadora de correntes - v3.3
1 - Converter u/v em velocidade + azimute
2 - Converter velocidade + azimute em u/v
3 - Calcular vetor médio de várias medições manuais
4 - Explicar conceitos usados no programa
0 - Sair

Encerrado.


## 18. Sequência recomendada de estudo

1. Execute as células de importação e definição das funções.
2. Modifique os exemplos numéricos e observe os resultados.
3. Teste direções cardeais e intercardeais.
4. Crie listas com correntes opostas e compare as médias vetorial e escalar.
5. Somente depois execute o menu interativo.

Dessa forma, o notebook serve ao mesmo tempo como programa, material de estudo de Python e resumo dos conceitos vetoriais empregados em oceanografia.